# Warehouse basics

Plain commands against the warehouse `ffe run` writes to. Read-only except the
last cell, which drops nothing unless you name a table.

In [1]:
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from ffe.io.sink import catalog, scan

WAREHOUSE = ROOT / "_ffe" / "warehouse"       # ffe run --workspace ./_ffe
pl.Config(fmt_str_lengths=40, tbl_rows=12)
cat = catalog(WAREHOUSE)

## What's in there

In [9]:
for (ns,) in cat.list_namespaces():
    for _, name in cat.list_tables(ns):
        t = cat.load_table(f"{ns}.{name}")
        s = t.current_snapshot()
        print(f"{ns}.{name:22} "
              f"rows={int(s.summary.get('total-records', 0)) if s else 0:>7}  "
              f"files={int(s.summary.get('total-data-files', 0)) if s else 0:>4}  "
              f"by={[f.name for f in t.spec().fields] or '(unpartitioned)'}")

bronze.msci_test              rows=    199  files=  21  by=['business_date']


## Read a table

In [10]:
TABLE = "bronze.msci_test"

df = scan(WAREHOUSE, TABLE)          # whole table -> polars
print(df.shape, df.columns)
df.head()

(199, 12) ['Sector_ID', 'Industry_Group_ID', 'Industry_ID', 'Sub_Industry_ID', 'Label_EN', 'Label_FR', '_src_line_no', 'business_date', '_src_file', '_job_id', '_spec_hash', '_ingested_at']


Sector_ID,Industry_Group_ID,Industry_ID,Sub_Industry_ID,Label_EN,Label_FR,_src_line_no,business_date,_src_file,_job_id,_spec_hash,_ingested_at
str,str,str,str,str,str,u32,date,str,str,str,datetime[μs]
"""10""","""1010""","""101010""","""10101010""","""Energy (Oil & Gas)""","""Énergie""",8,2026-08-17,"""taxonomy_20260817.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453443
"""15""","""1510""","""151040""","""15104020""","""Gold""","""Or""",9,2026-08-17,"""taxonomy_20260817.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453443
"""20""","""2010""","""201060""","""20106010""","""Industrial Machinery""","""Machines Industrielles""",10,2026-08-17,"""taxonomy_20260817.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453443
"""25""","""2510""","""251020""","""25102010""","""Automobiles""","""Automobiles""",11,2026-08-17,"""taxonomy_20260817.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453443
"""30""","""3020""","""302020""","""30202010""","""Packaged Foods & Meats""","""Produits Alimentaires""",12,2026-08-17,"""taxonomy_20260817.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453443


Same read, filter pushed into Iceberg so it only opens the files it needs.
Works partitioned or not — `plan_files()` says how many it opened.

In [12]:
FILTER = "business_date == '2026-08-18'"       # e.g. "business_date == '2026-08-17'" -- must be a real column

t = cat.load_table(TABLE)
q = t.scan(row_filter=FILTER) if FILTER else t.scan()
print("files opened:", len(list(q.plan_files())), "of", len(list(t.scan().plan_files())))
pl.from_arrow(q.to_arrow()).head()

files opened: 3 of 21


Sector_ID,Industry_Group_ID,Industry_ID,Sub_Industry_ID,Label_EN,Label_FR,_src_line_no,business_date,_src_file,_job_id,_spec_hash,_ingested_at
str,str,str,str,str,str,u32,date,str,str,str,datetime[μs]
"""20""","""2030""","""203020""","""20302010""","""Passenger Airlines""","""Compagnies Aériennes""",8,2026-08-18,"""taxonomy_20260818.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453741
"""25""","""2550""","""255030""","""25503020""","""Apparel Retail""","""Vêtements""",9,2026-08-18,"""taxonomy_20260818.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453741
"""30""","""3030""","""303010""","""30301010""","""Household Products""","""Produits Ménagers""",10,2026-08-18,"""taxonomy_20260818.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453741
"""35""","""3520""","""352020""","""35202010""","""Pharmaceuticals""","""Produits Pharmaceutiques""",11,2026-08-18,"""taxonomy_20260818.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453741
"""40""","""4030""","""403010""","""40301040""","""Property & Casualty Insurance""","""Assurance Dommages""",12,2026-08-18,"""taxonomy_20260818.zip!taxonomy_setup_202…","""fabd8179edd4""","""8aa5c7ae993eed3e""",2026-08-22 19:54:24.453741


## Check the files

In [5]:
files = cat.load_table(TABLE).inspect.files().to_pylist()

for r in files:
    p = Path(r["file_path"].replace("file://", ""))
    print(f"{'ok  ' if p.exists() else 'GONE'} {r['record_count']:>6} rows "
          f"{r['file_size_in_bytes']:>9} B  "
          f"{p.relative_to(ROOT) if p.is_relative_to(ROOT) else p}")

missing = [r for r in files if not Path(r["file_path"].replace("file://", "")).exists()]
print(f"\n{len(files)} files, {len(missing)} missing")

ok        5 rows      4994 B  _ffe/staging/efe1609d08e3/good/mock_taxonomy_data.zip_taxonomy_setup_20260823.txt.parquet
ok        3 rows      4895 B  _ffe/staging/efe1609d08e3/good/mock_taxonomy_data.zip_taxonomy_setup_20260824.txt.parquet
ok        4 rows      4833 B  _ffe/staging/efe1609d08e3/good/mock_taxonomy_data.zip_taxonomy_setup_20260825.txt.parquet

3 files, 0 missing


Data files live where the workers wrote them (`_ffe/staging/<job_id>/good/`) —
`add_files` registers them in place, so the warehouse dir holds metadata only.
Delete staging and the read fails with `FileNotFoundError`.

## Drop a table, if needed

`drop_table` removes the catalog entry only. `purge_table` also deletes the
Parquet — which for this project means deleting out of staging.

In [ ]:
DROP = None          # e.g. "bronze.msci_test" -- name it, then run this cell

if DROP and cat.table_exists(DROP):
    cat.drop_table(DROP)             # cat.purge_table(DROP) to delete the parquet too
    print("dropped", DROP)
else:
    print("nothing dropped")

dropped bronze.msci_test
